In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("datasets/dataset_final.csv")

In [3]:
df.columns

Index(['Domain_URL_Ratio', 'Count_www', 'Count_/', 'Path_Length', 'URL/Path',
       'Character_Repetition', 'Having_Path', 'Special_Char_Alphabet_Ratio',
       'ShannonEntropy', 'fd_length', 'Digit/Letter', 'Count_Digit',
       'FractalDimension', 'Kolmogorov_Complexity', 'Average_Word',
       'Base64_Pattern_Cnt', 'Count_Letter', 'Count_Https', 'Vowel/Consonant',
       'Count_Dot', 'Host_Precense_Of_Digit', 'Domain_Length_Of_URL',
       'Subdomain', 'Count_-', 'Uppercase_Lowercase_Ratio',
       'Longest_Word_in_Hostname', 'Count_Http', 'Count_Embed_Domain',
       'Tld_Length', 'use_of_ip_address', 'Longest_Word', 'Count_?',
       'Query_Length', 'Count_&', 'Count_;', 'Label'],
      dtype='object')

In [4]:
import pandas as pd
import time
import psutil
import os
import joblib

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.base import clone

# ================= DATA =================
X = df.drop('Label', axis=1)
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ================= MODELLER =================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(),
    "Linear SVM": LinearSVC(),
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

# ================= HİBRİT MODEL =================
stacking_model = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss')),
        ('svm', LinearSVC())
    ],
    final_estimator=LogisticRegression(),
    n_jobs=-1
)

models["Stacking (Hybrid)"] = stacking_model

# ================= MODEL KLASÖRÜ =================
os.makedirs("models", exist_ok=True)

# ================= EĞİTİM =================
results = []
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
process = psutil.Process()

for name, model in models.items():
    try:
        print(f">>> {name} eğitiliyor...")

        current_model = clone(model)

        start_time = time.time()
        start_mem = process.memory_info().rss / (1024 ** 2)

        current_model.fit(X_train, y_train)
        train_time = time.time() - start_time

        pred_start = time.time()
        y_pred = current_model.predict(X_test)
        pred_time = time.time() - pred_start

        end_mem = process.memory_info().rss / (1024 ** 2)
        mem_used = end_mem - start_mem
        cpu_used = psutil.cpu_percent(interval=0.1)

        row = {
            "Model": name,
            "Test_Accuracy": accuracy_score(y_test, y_pred),
            "Test_Precision": precision_score(y_test, y_pred),
            "Test_Recall": recall_score(y_test, y_pred),
            "Test_F1": f1_score(y_test, y_pred),
            "Train_Time (s)": round(train_time, 4),
            "Predict_Time (s)": round(pred_time, 4),
            "Memory_Usage (MB)": round(mem_used, 2),
            "CPU_Usage (%)": cpu_used
        }

        # ROC-AUC
        try:
            if hasattr(current_model, "predict_proba"):
                y_prob = current_model.predict_proba(X_test)[:, 1]
            else:
                y_prob = current_model.decision_function(X_test)
            row["Test_ROC_AUC"] = roc_auc_score(y_test, y_prob)
        except:
            row["Test_ROC_AUC"] = None

        print(f"    {name} için Cross-Validation...")
        cv_scores = cross_validate(clone(model), X, y, cv=5, scoring=scoring, n_jobs=-1)

        row["CV_Accuracy"] = cv_scores["test_accuracy"].mean()
        row["CV_F1"] = cv_scores["test_f1"].mean()
        row["CV_ROC_AUC"] = cv_scores["test_roc_auc"].mean()

        results.append(row)

        # ================= MODEL KAYDETME =================
        model_filename = f"models/{name.replace(' ', '_').replace('(', '').replace(')', '')}.joblib"
        joblib.dump(current_model, model_filename)
        print(f"{name} modeli kaydedildi: {model_filename}")

        print(f"--- {name} tamamlandı ---\n")

    except Exception as e:
        print(f"!!! {name} hatası: {e}")

# ================= SONUÇ =================
results_table = pd.DataFrame(results).sort_values("Test_Accuracy", ascending=False)
results_table = results_table.reset_index(drop=True)
results_table.index = results_table.index + 1

print("\n=== TÜM MODELLER PERFORMANS TABLOSU ===")
print(results_table)

>>> Logistic Regression eğitiliyor...
    Logistic Regression için Cross-Validation...
Logistic Regression modeli kaydedildi: models/Logistic_Regression.joblib
--- Logistic Regression tamamlandı ---

>>> Random Forest eğitiliyor...
    Random Forest için Cross-Validation...
Random Forest modeli kaydedildi: models/Random_Forest.joblib
--- Random Forest tamamlandı ---

>>> Decision Tree eğitiliyor...
    Decision Tree için Cross-Validation...
Decision Tree modeli kaydedildi: models/Decision_Tree.joblib
--- Decision Tree tamamlandı ---

>>> Linear SVM eğitiliyor...
    Linear SVM için Cross-Validation...
Linear SVM modeli kaydedildi: models/Linear_SVM.joblib
--- Linear SVM tamamlandı ---

>>> Naive Bayes eğitiliyor...
    Naive Bayes için Cross-Validation...
Naive Bayes modeli kaydedildi: models/Naive_Bayes.joblib
--- Naive Bayes tamamlandı ---

>>> KNN eğitiliyor...
    KNN için Cross-Validation...
KNN modeli kaydedildi: models/KNN.joblib
--- KNN tamamlandı ---

>>> Gradient Boosting eği

In [ ]:
################################

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("datasets/features_dataset.csv")

MODEL_BEKLENEN_SIRA = [
    'Domain_URL_Ratio', 'Count_www', 'Count_/', 'Path_Length', 'URL/Path',
    'Character_Repetition', 'Having_Path', 'Special_Char_Alphabet_Ratio',
    'ShannonEntropy', 'fd_length', 'Digit/Letter', 'Count_Digit',
    'FractalDimension', 'Kolmogorov_Complexity', 'Average_Word',
    'Base64_Pattern_Cnt', 'Count_Letter', 'Count_Https', 'Vowel/Consonant',
    'Count_Dot', 'Host_Precense_Of_Digit', 'Domain_Length_Of_URL', 'Subdomain',
    'Count_-', 'Uppercase_Lowercase_Ratio', 'Longest_Word_in_Hostname',
    'Count_Http', 'Count_Embed_Domain', 'Tld_Length', 'use_of_ip_address',
    'Longest_Word', 'Count_?', 'Query_Length', 'Count_&', 'Count_;'
]

df['Label'] = df['Label'].map({'Phishing': 1, 'Legitimate': 0})

import joblib
model = joblib.load("models/XGBoost.joblib")

# 5 Legitimate örnek
legit = df[df['Label'] == 0].iloc[0:5]
X = legit[MODEL_BEKLENEN_SIRA]
print("5 Legitimate tahmin:", model.predict(X))
print("5 Legitimate olasılık:\n", model.predict_proba(X))

# Instagram'ın değerlerini elle ver
instagram = pd.DataFrame([{
    'Domain_URL_Ratio': 0.531250, 'Count_www': 1.0, 'Count_/': 3.0,
    'Path_Length': 1.0, 'URL/Path': 32.0, 'Character_Repetition': 4.0,
    'Having_Path': 0.0, 'Special_Char_Alphabet_Ratio': 0.333333,
    'ShannonEntropy': 4.179229, 'fd_length': 0.0, 'Digit/Letter': 0.0,
    'Count_Digit': 0.0, 'FractalDimension': 0.625000,
    'Kolmogorov_Complexity': 1.250000, 'Average_Word': 4.0,
    'Base64_Pattern_Cnt': 1.0, 'Count_Letter': 24.0, 'Count_Https': 1.0,
    'Vowel/Consonant': 0.263158, 'Count_Dot': 2.0,
    'Host_Precense_Of_Digit': 0.0, 'Domain_Length_Of_URL': 17.0,
    'Subdomain': 1.0, 'Count_-': 0.0, 'Uppercase_Lowercase_Ratio': 0.0,
    'Longest_Word_in_Hostname': 9.0, 'Count_Http': 1.0,
    'Count_Embed_Domain': 1.0, 'Tld_Length': 3.0, 'use_of_ip_address': 0.0,
    'Longest_Word': 9.0, 'Count_?': 1.0, 'Query_Length': 5.0,
    'Count_&': 0.0, 'Count_;': 0.0
}])[MODEL_BEKLENEN_SIRA]

print("\nInstagram tahmini:", model.predict(instagram))
print("Instagram olasılık:", model.predict_proba(instagram))

5 Legitimate tahmin: [0 0 0 0 0]
5 Legitimate olasılık:
 [[9.9984825e-01 1.5174483e-04]
 [9.9839836e-01 1.6016603e-03]
 [9.9794394e-01 2.0560494e-03]
 [9.9911404e-01 8.8593405e-04]
 [9.9584079e-01 4.1592214e-03]]

Instagram tahmini: [1]
Instagram olasılık: [[9.5367432e-07 9.9999905e-01]]


In [8]:
from sklearn.metrics.pairwise import euclidean_distances
import numpy as np

# Eğitim setindeki Legitimate örnekler
legit_df = df[df['Label'] == 0][MODEL_BEKLENEN_SIRA]

# Instagram feature vektörü
insta = instagram.values

# En yakın 5 Legitimate örneği bul
dists = euclidean_distances(insta, legit_df)
closest_idx = np.argsort(dists[0])[:5]
print("Instagram'a en yakın 5 Legitimate örnek:")
print(legit_df.iloc[closest_idx])

# Aynısını Phishing için de yap
phish_df = df[df['Label'] == 1][MODEL_BEKLENEN_SIRA]
dists_p = euclidean_distances(insta, phish_df)
closest_idx_p = np.argsort(dists_p[0])[:5]
print("\nInstagram'a en yakın 5 Phishing örnek:")
print(phish_df.iloc[closest_idx_p])

# Mesafe karşılaştırması
print(f"\nEn yakın Legitimate mesafesi: {dists[0][closest_idx[0]]:.4f}")
print(f"En yakın Phishing mesafesi:   {dists_p[0][closest_idx_p[0]]:.4f}")

Instagram'a en yakın 5 Legitimate örnek:
        Domain_URL_Ratio  Count_www  Count_/  Path_Length  URL/Path  \
950             0.724138          1        2            0      29.0   
79517           0.724138          1        2            0      29.0   
278247          0.724138          1        2            0      29.0   
132820          0.724138          1        2            0      29.0   
162299          0.724138          1        2            0      29.0   

        Character_Repetition  Having_Path  Special_Char_Alphabet_Ratio  \
950                        4            0                      0.26087   
79517                      4            0                      0.26087   
278247                     4            0                      0.26087   
132820                     4            0                      0.26087   
162299                     4            0                      0.26087   

        ShannonEntropy  fd_length  ...  Longest_Word_in_Hostname  Count_Http  \
950    

In [10]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

model = joblib.load("models/XGBoost.joblib")

df = pd.read_csv("datasets/dataset_final.csv")

MODEL_BEKLENEN_SIRA = [
    'Domain_URL_Ratio', 'Count_www', 'Count_/', 'Path_Length', 'URL/Path',
    'Character_Repetition', 'Having_Path', 'Special_Char_Alphabet_Ratio',
    'ShannonEntropy', 'fd_length', 'Digit/Letter', 'Count_Digit',
    'FractalDimension', 'Kolmogorov_Complexity', 'Average_Word',
    'Base64_Pattern_Cnt', 'Count_Letter', 'Count_Https', 'Vowel/Consonant',
    'Count_Dot', 'Host_Precense_Of_Digit', 'Domain_Length_Of_URL', 'Subdomain',
    'Count_-', 'Uppercase_Lowercase_Ratio', 'Longest_Word_in_Hostname',
    'Count_Http', 'Count_Embed_Domain', 'Tld_Length', 'use_of_ip_address',
    'Longest_Word', 'Count_?', 'Query_Length', 'Count_&', 'Count_;'
]

X = df[MODEL_BEKLENEN_SIRA]
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

probs = model.predict_proba(X_test)[:, 1]

print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Phishing_caught%':>17}")
for t in np.arange(0.50, 0.96, 0.05):
    preds = (probs >= t).astype(int)
    p = precision_score(y_test, preds)
    r = recall_score(y_test, preds)
    f = f1_score(y_test, preds)
    caught = preds[y_test == 1].mean()
    print(f"{t:>10.2f} {p:>10.4f} {r:>10.4f} {f:>10.4f} {caught:>17.4f}")

 Threshold  Precision     Recall         F1  Phishing_caught%
      0.50     0.9995     0.9959     0.9977            0.9959
      0.55     0.9996     0.9958     0.9977            0.9958
      0.60     0.9997     0.9957     0.9977            0.9957
      0.65     0.9997     0.9956     0.9976            0.9956
      0.70     0.9997     0.9955     0.9976            0.9955
      0.75     0.9997     0.9954     0.9975            0.9954
      0.80     0.9997     0.9953     0.9975            0.9953
      0.85     0.9997     0.9949     0.9973            0.9949
      0.90     0.9998     0.9947     0.9972            0.9947
      0.95     0.9999     0.9942     0.9970            0.9942
